In [7]:
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd

from src.features import create_features

COLUMNS = ['AQI', 'PM2.5', 'PM10', 'NO2', 'CO', 'O3']

def load_processed(path='data/processed/hyderabad_aqi.csv'):
    frame = pd.read_csv(Path(path), parse_dates=['Date'], index_col='Date')
    return frame.sort_index()[COLUMNS]



In [8]:
hyd = load_processed('..\data\processed\hyderabad_aqi.csv')
model_data = create_features(hyd)
print(model_data.shape)
display(model_data.head())

(1656, 23)


,AQI,PM2.5,PM10,NO2,CO,O3,month,day_of_week,is_weekend,aqi_lag_1,...,aqi_lag_2,aqi_lag_3,aqi_lag_7,aqi_lag_14,PM2.5_lag_1,tomorrow_aqi,PM10_lag_1,NO2_lag_1,CO_lag_1,O3_lag_1
Date,,,,,,,,,,,,,,,,,,,,,
2015-09-03,263.0,96.48,85.8875,13.47,0.78,26.75,9,3,0,136.0,...,124.0,97.0,128.0,214.0,90.29,237.0,93.0000,11.40,0.67,22.47
2015-09-04,237.0,161.72,78.7750,14.42,0.85,27.06,9,4,0,263.0,...,136.0,124.0,253.0,316.0,96.48,305.0,85.8875,13.47,0.78,26.75
2015-09-07,114.0,48.83,30.7900,22.54,0.86,29.29,9,0,0,237.0,...,305.0,237.0,97.0,88.0,59.42,179.0,64.5500,22.56,0.93,26.32
2015-09-08,179.0,91.82,32.9400,28.93,0.48,27.04,9,1,0,114.0,...,237.0,305.0,124.0,173.0,48.83,162.0,30.7900,22.54,0.86,29.29
2015-09-09,162.0,35.56,40.8100,31.15,0.57,22.48,9,2,0,179.0,...,114.0,237.0,136.0,114.0,91.82,76.0,32.9400,28.93,0.48,27.04


In [9]:
target = 'tomorrow_aqi'
feature_columns = [c for c in model_data.columns if c != target]

train_end = int(len(model_data) * 0.70)

validation_end = int(len(model_data) * 0.85)

train = model_data.iloc[:train_end]

validation = model_data.iloc[train_end:validation_end]

test = model_data.iloc[validation_end:]

assert train.index.max() < validation.index.min() < test.index.min()

print(len(train), len(validation), len(test))



1159 248 249


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_val = validation[target]
baseline_today = validation['AQI']
baseline_week = validation['aqi_lag_7']

def show_scores(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5
    print(f'{name}: MAE={mae:.2f}, RMSE={rmse:.2f}')

show_scores('Tomorrow equals today', y_val, baseline_today)
show_scores('Same weekday last week', y_val, baseline_week)


Tomorrow equals today: MAE=8.33, RMSE=10.89
Same weekday last week: MAE=17.54, RMSE=21.65
